# 15 — RQ2 blind audit analysis

Merges the completed 240 audit labels with the hidden key, then computes **sampling-weighted** 
precision / recall / F1 for each construct (overall, manosphere, and comparison communities) with 
10,000-sample bootstrap 95% CIs, and exports every model–annotator disagreement (FP/FN) for the 
qualitative error read. The nested 1B rule (1B only where 1A) is applied to predictions throughout.

Reads from and writes to `outputs/rq2/`. Requires the labelled audit file and key from notebook 14.


In [ ]:
# Merge the 240 completed audit labels with the hidden prediction key,
# then compute sampling-weighted agreement metrics.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = str(PROJECT_ROOT / "outputs" / "rq2")
AUDIT   = f"{RESULTS}/rq2_final_audit_240_MERGED.csv"
CONSTRUCTS = ["1A", "1B", "2"]

def load(path):
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="cp1252")
    for c in ["pred_1A","pred_1B","pred_2","manual_1A","manual_1B","manual_2"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    df["pred_1B"] = ((df["pred_1A"]==1) & (df["pred_1B"]==1)).astype("Int64")   # 1B nested in 1A
    return df

df = load(AUDIT)
rows = []
for con in CONSTRUCTS:
    p, m, pr = f"pred_{con}", f"manual_{con}", f"prob_{con}"
    dis = df[df[p] != df[m]].copy()
    dis["construct"]  = con
    dis["error_type"] = dis[m].map({1: "FN", 0: "FP"})   # FN=you 1/model 0, FP=model 1/you 0
    dis["prob"]       = dis[pr]
    dis["pred"]       = dis[p]
    dis["manual"]     = dis[m]
    rows.append(dis[[
        "community_group","construct","error_type","prob",
        "pred","manual","pred_1A","manual_1A","manual_2",
        "audit_id","channel_name","text_clean"
    ]])

out = pd.concat(rows, ignore_index=True)
out = out.sort_values(["construct","community_group","error_type","prob"],
                      ascending=[True,True,True,False])

path_out = f"{RESULTS}/audit_new_four_FN_FP_by_construct_community.csv"
out.to_csv(path_out, index=False, encoding="utf-8-sig")
print("saved:", path_out, "|", len(out), "disagreement rows")
print(out.groupby(["construct","community_group","error_type"]).size())

In [ ]:
# Load the completed audit labels (upload the file when prompted) and the prediction key.
import io
from pathlib import Path
import numpy as np, pandas as pd
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = PROJECT_ROOT / "outputs" / "rq2"
KEY_PATH    = RESULTS / "rq2_final_audit_240_KEY_DO_NOT_OPEN.csv"
MERGED_PATH = RESULTS / "rq2_final_audit_240_MERGED.csv"
OUT_DIR     = RESULTS / "audit_new_four"; OUT_DIR.mkdir(parents=True, exist_ok=True)

COMPARISON = ["male_oriented_wellbeing","mainstream_self_improvement","unrelated_control"]
MANUAL = ["manual_1A","manual_1B","manual_2"]
N_BOOT, SEED = 10000, 42

# Read a CSV (any encoding) or an Excel export, from bytes or a path.
def read_bytes(name, data):
    if data[:2] == b"PK" or name.lower().endswith((".xlsx",".xls")):
        return pd.read_excel(io.BytesIO(data))
    for enc in ["utf-8-sig","utf-16","cp1252","latin-1"]:
        try: return pd.read_csv(io.BytesIO(data), encoding=enc)
        except (UnicodeDecodeError, UnicodeError): continue
    raise ValueError(f"Could not read {name}")

def read_path(p):
    p = Path(p)
    if p.suffix.lower() in (".xlsx",".xls"): return pd.read_excel(p)
    for enc in ["utf-8-sig","utf-16","cp1252","latin-1"]:
        try: return pd.read_csv(p, encoding=enc)
        except (UnicodeDecodeError, UnicodeError): continue
    raise ValueError(f"Could not read {p}")

# Read the completed labels file produced during annotation.
LABELLED_PATH = RESULTS / "rq2_final_audit_240_labelled.csv"
lab = read_path(LABELLED_PATH)
key = read_path(KEY_PATH)

# Tidy column names and drop any stray index columns.
for d in (lab, key):
    d.columns = d.columns.astype(str).str.strip()
lab = lab.loc[:, ~lab.columns.str.startswith("Unnamed:")]
key = key.loc[:, ~key.columns.str.startswith("Unnamed:")]
lab["audit_id"] = lab["audit_id"].astype(str).str.strip()
key["audit_id"] = key["audit_id"].astype(str).str.strip()

# Both files hold the same 240 comments, one row each.
assert len(lab)==240 and len(key)==240
assert lab["audit_id"].is_unique and key["audit_id"].is_unique
assert set(lab["audit_id"])==set(key["audit_id"])

# Manual labels must be clean 0/1 values.
for c in MANUAL:
    lab[c] = pd.to_numeric(lab[c], errors="coerce")
    assert lab[c].notna().all() and lab[c].isin([0,1]).all()
    lab[c] = lab[c].astype(int)

# A comment can only be 1B if it is also 1A.
assert not ((lab["manual_1B"]==1)&(lab["manual_1A"]==0)).any()

# Attach the manual labels to the key and save the combined file.
if "notes" not in lab.columns: lab["notes"] = pd.NA
key_clean = key.drop(columns=MANUAL+["notes"], errors="ignore")
merged = key_clean.merge(lab[["audit_id",*MANUAL,"notes"]], on="audit_id", how="left", validate="one_to_one")
assert len(merged)==240 and merged[MANUAL].notna().all().all()
merged.to_csv(MERGED_PATH, index=False, encoding="utf-8-sig")
print("merged saved:", MERGED_PATH)
print("\nmanual positives by community:\n", merged.groupby("community_group")[MANUAL].sum())

# Weighted confusion counts and the precision/recall/F1 they imply.
def w_conf(yt,yp,w):
    return (np.sum(w*((yt==1)&(yp==1))), np.sum(w*((yt==0)&(yp==1))),
            np.sum(w*((yt==1)&(yp==0))), np.sum(w*((yt==0)&(yp==0))))
def prf(tp,fp,fn):
    p = tp/(tp+fp) if tp+fp>0 else np.nan
    r = tp/(tp+fn) if tp+fn>0 else np.nan
    f = 2*p*r/(p+r) if p==p and r==r and p+r>0 else np.nan
    return p,r,f

df = merged.copy()
for c in ["pred_1A","pred_1B","pred_2",*MANUAL]:
    df[c] = pd.to_numeric(df[c], errors="raise").astype(int)

# Apply the same 1B-within-1A rule to the predictions.
df["pred_1B"] = ((df["pred_1A"]==1)&(df["pred_1B"]==1)).astype(int)

# Sampling weights undo the deliberate oversampling of positive strata.
w_all = pd.to_numeric(df["sampling_weight"], errors="raise").to_numpy()
specs = [("1A","manual_1A","pred_1A"),("1B","manual_1B","pred_1B"),("2","manual_2","pred_2")]

# Performance overall, in the manosphere, and across the comparison communities.
scopes = {"overall": np.ones(len(df),bool),
          "manosphere": (df["community_group"]=="manosphere_adjacent").to_numpy(),
          "comparison": df["community_group"].isin(COMPARISON).to_numpy()}

rng = np.random.default_rng(SEED); rows=[]
for scope,mask in scopes.items():
    for lb,mc,pc in specs:
        yt=df[mc].to_numpy()[mask]; yp=df[pc].to_numpy()[mask]; w=w_all[mask]; n=len(yt)
        tp,fp,fn,tn=w_conf(yt,yp,w); p,r,f=prf(tp,fp,fn)
        # Bootstrap the audited comments for 95% intervals.
        bp,br,bf=[],[],[]
        for _ in range(N_BOOT):
            idx=rng.integers(0,n,n)
            pp,rr,ff=prf(*w_conf(yt[idx],yp[idx],w[idx])[:3]); bp.append(pp); br.append(rr); bf.append(ff)
        q=lambda a:(round(np.nanpercentile(a,2.5),3),round(np.nanpercentile(a,97.5),3))
        rows.append({"scope":scope,"construct":lb,"precision":round(p,3),"precision_CI":q(bp),
                     "recall":round(r,3),"recall_CI":q(br),"f1":round(f,3),"f1_CI":q(bf),
                     "w_TP":round(tp,1),"w_FP":round(fp,1),"w_FN":round(fn,1),"w_TN":round(tn,1)})
metrics=pd.DataFrame(rows)
metrics.to_csv(OUT_DIR/"new_four_AUDIT_weighted_metrics.csv", index=False)
print("\nweighted metrics:\n", metrics.to_string(index=False))

# Every comment where the model and the annotator disagree, for the qualitative read.
erows=[]
for con in ["1A","1B","2"]:
    pc,mc,prc=f"pred_{con}",f"manual_{con}",f"prob_{con}"
    dis=df[df[pc]!=df[mc]].copy()
    dis["construct"]=con
    dis["error_type"]=dis[mc].map({1:"FN",0:"FP"})   # FN = annotator 1, model 0; FP = model 1, annotator 0
    dis["prob"]=dis[prc]; dis["pred"]=dis[pc]; dis["manual"]=dis[mc]
    erows.append(dis[["community_group","construct","error_type","prob","pred","manual",
                      "pred_1A","manual_1A","manual_2","audit_id","channel_name","text_clean"]])
out=pd.concat(erows, ignore_index=True).sort_values(
    ["construct","community_group","error_type","prob"], ascending=[True,True,True,False])
out.to_csv(RESULTS/"audit_new_four_FN_FP_by_construct_community.csv", index=False, encoding="utf-8-sig")
print(f"\ndisagreements: {len(out)} rows")
print(out.groupby(["construct","community_group","error_type"]).size())

In [ ]:
pd.set_option("display.max_colwidth", None)
for (con, comm, etype), grp in out.groupby(["construct","community_group","error_type"], sort=False):
    print("\n"+"="*100); print(f"construct {con} | {comm} | {etype} (n={len(grp)})"); print("="*100)
    for i, r in enumerate(grp.itertuples(), 1):
        print(f"\n[{con}-{etype}{i}] {r.channel_name} | p={r.prob:.2f} | pred_1A={r.pred_1A} manual_1A={r.manual_1A} manual_2={r.manual_2}")
        print(f'    "{str(r.text_clean).strip()}"')

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = PROJECT_ROOT / "outputs" / "rq2"
df = pd.read_csv(R/"rq2_final_audit_240_MERGED.csv", encoding="utf-8-sig")
for c in ["pred_1A","pred_1B","pred_2","manual_1A","manual_1B","manual_2"]:
    df[c] = pd.to_numeric(df[c], errors="raise").astype(int)
df["pred_1B"] = ((df.pred_1A==1)&(df.pred_1B==1)).astype(int)

comm_order = ["manosphere_adjacent","male_oriented_wellbeing","mainstream_self_improvement","unrelated_control"]
comm_lab = ["Manosphere","Wellbeing","Mainstream","Control"]
cons = ["1A","1B","2"]
con_lab = ["1A\nGrievance","1B\nWeaponised","2\nConspiracy"]

def grid(kind):
    M = np.zeros((3,4), int)
    for i,con in enumerate(cons):
        for j,comm in enumerate(comm_order):
            s = df[df.community_group==comm]
            m,p = s[f"manual_{con}"], s[f"pred_{con}"]
            M[i,j] = int(((m==0)&(p==1)).sum()) if kind=="FP" else int(((m==1)&(p==0)).sum())
    return M

FP, FN = grid("FP"), grid("FN")
vmax = max(FP.max(), FN.max())
navy = LinearSegmentedColormap.from_list("nv", ["#f7f9fc","#c9d7ec","#6b8fc4","#1f3864"])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, M, title in [(axes[0],FP,"False positives"),(axes[1],FN,"False negatives")]:
    ax.imshow(M, cmap=navy, vmin=0, vmax=vmax)
    for i in range(3):
        for j in range(4):
            ax.text(j, i, M[i,j], ha="center", va="center", fontsize=15, fontweight="bold",
                    color="white" if M[i,j] > vmax*0.55 else "#1f3864")
    ax.set_xticks(range(4), comm_lab, fontsize=9.5)
    ax.set_yticks(range(3), con_lab, fontsize=9.5)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    ax.set_xticks(np.arange(-.5,4,1), minor=True); ax.set_yticks(np.arange(-.5,3,1), minor=True)
    ax.grid(which="minor", color="white", lw=2); ax.tick_params(which="both", length=0)
    for sp in ax.spines.values(): sp.set_visible(False)
fig.suptitle("Audit errors by construct and community (counts)", fontsize=12.5, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(R/"rq2_audit_fpfn.png", dpi=350, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ---- 1. Load the completed 240-comment audit dataset ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = PROJECT_ROOT / "outputs" / "rq2"
AUDIT_FILE = RESULTS / "rq2_final_audit_240_MERGED.csv"

merged = pd.read_csv(AUDIT_FILE, encoding="utf-8-sig")
merged.columns = merged.columns.astype(str).str.strip()

# ---- 2. Check that this is the right file ----
required = {
    "community_group",
    "pred_1A", "pred_1B", "pred_2",
    "manual_1A", "manual_1B", "manual_2"
}

missing = required - set(merged.columns)

if missing:
    raise ValueError(
        "This is not the completed merged audit file. "
        f"Missing columns: {sorted(missing)}\n\n"
        f"Columns available: {merged.columns.tolist()}"
    )

# Ensure labels are numeric 0/1.
label_cols = [
    "pred_1A", "pred_1B", "pred_2",
    "manual_1A", "manual_1B", "manual_2"
]

for col in label_cols:
    merged[col] = pd.to_numeric(merged[col], errors="raise").astype(int)

# Preserve the nested-label rule used in the audit analysis.
merged["pred_1B"] = (
    (merged["pred_1A"] == 1) & (merged["pred_1B"] == 1)
).astype(int)

# ---- 3. Define groups and constructs ----
COMPARISON = [
    "male_oriented_wellbeing",
    "mainstream_self_improvement",
    "unrelated_control"
]

constructs = {
    "1A · Grievance": ("manual_1A", "pred_1A"),
    "1B · Weaponised\nvictimhood": ("manual_1B", "pred_1B"),
    "2 · Conspiratorial\nframing": ("manual_2", "pred_2")
}

groups = {
    "Manosphere": merged["community_group"].eq("manosphere_adjacent"),
    "Comparison\n(pooled)": merged["community_group"].isin(COMPARISON)
}

# ---- 4. Count unweighted audit-sample errors ----
plot_rows = []

for group_name, mask in groups.items():
    d = merged.loc[mask]

    for construct_name, (manual_col, pred_col) in constructs.items():
        false_positive = int(((d[manual_col] == 0) & (d[pred_col] == 1)).sum())
        false_negative = int(((d[manual_col] == 1) & (d[pred_col] == 0)).sum())

        plot_rows.extend([
            {
                "Audit group": group_name,
                "Construct": construct_name,
                "Error type": "False positive",
                "Count": false_positive
            },
            {
                "Audit group": group_name,
                "Construct": construct_name,
                "Error type": "False negative",
                "Count": false_negative
            }
        ])

errors = pd.DataFrame(plot_rows)

print(errors.pivot(
    index="Construct",
    columns=["Audit group", "Error type"],
    values="Count"
))

# ---- 5. Plot ----
sns.set_theme(style="whitegrid", font_scale=1.05)

palette = {
    "False positive": "#C65D4B",
    "False negative": "#4F81BD"
}

construct_order = list(constructs.keys())
error_order = ["False positive", "False negative"]
group_order = ["Manosphere", "Comparison\n(pooled)"]

fig, axes = plt.subplots(
    1, 2,
    figsize=(10.5, 4.7),
    sharey=False
)

for ax, group_name in zip(axes, group_order):
    d = errors.loc[errors["Audit group"] == group_name]

    sns.barplot(
        data=d,
        x="Construct",
        y="Count",
        hue="Error type",
        order=construct_order,
        hue_order=error_order,
        palette=palette,
        errorbar=None,
        ax=ax
    )

    ax.set_title(group_name, weight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(
        "Number of audit-sample errors"
        if group_name == "Manosphere"
        else ""
    )
    ax.set_ylim(0, d["Count"].max() + 2)

    if ax.legend_ is not None:
        ax.legend_.remove()

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for container in ax.containers:
        ax.bar_label(container, padding=3, fontsize=10, weight="bold")

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.03)
)

fig.suptitle(
    "Types of classification error in the stratified transfer audit",
    y=1.11,
    weight="bold"
)

fig.text(
    0.5,
    -0.03,
    "Unweighted counts are descriptive of the audit sample, not population error rates.",
    ha="center",
    fontsize=9
)

plt.tight_layout()

output_file = RESULTS / "transfer_audit_error_counts.png"
plt.savefig(output_file, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {output_file}")

In [ ]:
import re
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = PROJECT_ROOT / "outputs" / "rq2" / "audit_new_four"
m = pd.read_csv(R/"new_four_AUDIT_weighted_metrics.csv")
def ci_bounds(s):
    n = re.findall(r"\d+\.\d+", str(s)); return float(n[0]), float(n[1])
m[["f1_lo","f1_hi"]] = m["f1_CI"].apply(lambda s: pd.Series(ci_bounds(s)))

con = [("1A","1A"),("1B","1B"),("2","2")]
scopes = [("manosphere","Manosphere","#1f3864"),("comparison","Comparison (pooled)","#c0562f")]
ypos = {"1A":2,"1B":1,"2":0}; dodge=0.17

plt.rcParams.update({"font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],"font.size":12})
fig, ax = plt.subplots(figsize=(8,4.6))
for si,(sk,sl,col) in enumerate(scopes):
    sub = m[m["scope"]==sk].set_index("construct")
    for ck,_ in con:
        r = sub.loc[ck]; y = ypos[ck] + (dodge if si==0 else -dodge)
        ax.plot([r["f1_lo"],r["f1_hi"]],[y,y],color=col,lw=2.6,solid_capstyle="round",zorder=2,clip_on=False)
        ax.scatter(r["f1"],y,s=70,color=col,edgecolor="white",lw=1.1,zorder=3,clip_on=False)
        off = 0.15 if si==0 else -0.15; va = "bottom" if si==0 else "top"
        ax.text(r["f1"], y+off, f"{r['f1']:.2f}", ha="center", va=va, fontsize=10.5, color=col, fontweight="bold")
ax.set_yticks(list(ypos.values())); ax.set_yticklabels([lab for _,lab in con], fontsize=12)
ax.set_ylim(-0.7,2.7); ax.set_xlim(0,1.02); ax.set_xticks([0,.25,.5,.75,1])
ax.set_xlabel("F1 score", fontsize=11, labelpad=8)
ax.grid(axis="x",color="#E6E9EC",lw=0.9); ax.set_axisbelow(True)
for sp in ("top","right","left"): ax.spines[sp].set_visible(False)
ax.tick_params(length=0)

fig.suptitle("Audit F1 agreement between model and human labels with 95% CI",
             fontsize=12.5, fontweight="bold", y=0.97)
handles=[plt.Line2D([0],[0],marker="o",color="w",markerfacecolor=col,markersize=11,label=sl) for _,sl,col in scopes]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5,0.01), ncol=2, frameon=False, fontsize=11.5)
fig.subplots_adjust(top=0.89, bottom=0.24)
fig.savefig(R/"rq2_audit_f1.png", dpi=400, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
import re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES = PROJECT_ROOT / "outputs" / "rq2"
R = RES/"audit_new_four"
m = pd.read_csv(R/"new_four_AUDIT_weighted_metrics.csv")
def ci_bounds(s):
    n = re.findall(r"\d+\.\d+", str(s)); return float(n[0]), float(n[1])
m[["f1_lo","f1_hi"]] = m["f1_CI"].apply(lambda s: pd.Series(ci_bounds(s)))

merged = pd.read_csv(RES/"rq2_final_audit_240_MERGED.csv", encoding="utf-8-sig")
for c in ["manual_1A","manual_1B","manual_2"]:
    merged[c] = pd.to_numeric(merged[c], errors="raise").astype(int)
COMP = ["male_oriented_wellbeing","mainstream_self_improvement","unrelated_control"]
def support(sk, con):
    mask = merged.community_group.eq("manosphere_adjacent") if sk=="manosphere" else merged.community_group.isin(COMP)
    return int(merged.loc[mask, f"manual_{con}"].sum())

con = [("1A","1A"),("1B","1B"),("2","2")]
scopes = [("manosphere","Manosphere","#1f3864"),("comparison","Comparison (pooled)","#c0562f")]
ypos = {"1A":2,"1B":1,"2":0}; dodge=0.18; bh=0.30

plt.rcParams.update({"font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],"font.size":12})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.9), gridspec_kw={"width_ratios":[2.5,1]}, sharey=True)

# left: F1 forest
for si,(sk,sl,col) in enumerate(scopes):
    sub = m[m["scope"]==sk].set_index("construct")
    for ck,_ in con:
        r = sub.loc[ck]; y = ypos[ck] + (dodge if si==0 else -dodge)
        ax1.plot([r["f1_lo"],r["f1_hi"]],[y,y],color=col,lw=2.6,solid_capstyle="round",zorder=2,clip_on=False)
        ax1.scatter(r["f1"],y,s=70,color=col,edgecolor="white",lw=1.1,zorder=3,clip_on=False)
        off=0.15 if si==0 else -0.15; va="bottom" if si==0 else "top"
        ax1.text(r["f1"], y+off, f"{r['f1']:.2f}", ha="center", va=va, fontsize=10, color=col, fontweight="bold")
ax1.set_yticks(list(ypos.values())); ax1.set_yticklabels([lab for _,lab in con], fontsize=12)
ax1.set_ylim(-0.7,2.7); ax1.set_xlim(0,1.02); ax1.set_xticks([0,.25,.5,.75,1])
ax1.set_xlabel("F1 score", fontsize=11, labelpad=8)
ax1.grid(axis="x",color="#E6E9EC",lw=0.9); ax1.set_axisbelow(True)
for sp in ("top","right","left"): ax1.spines[sp].set_visible(False)
ax1.tick_params(length=0)

# right: positive-case counts
maxn = 0
for si,(sk,sl,col) in enumerate(scopes):
    for ck,_ in con:
        n = support(sk, ck); maxn = max(maxn, n)
        y = ypos[ck] + (dodge if si==0 else -dodge)
        ax2.barh(y, n, height=bh, color=col, alpha=0.9)
        ax2.text(n+0.5, y, f"{n}", va="center", ha="left", fontsize=9.5, color=col, fontweight="bold")
ax2.set_xlim(0, maxn*1.18)
ax2.set_xlabel("Positive cases (n)", fontsize=11, labelpad=8)
ax2.grid(axis="x",color="#E6E9EC",lw=0.9); ax2.set_axisbelow(True)
for sp in ("top","right","left"): ax2.spines[sp].set_visible(False)
ax2.tick_params(length=0)

fig.suptitle("Audit F1 (95% CI) and positive-case counts by construct", fontsize=12.5, fontweight="bold", y=0.98)
handles=[plt.Line2D([0],[0],marker="o",color="w",markerfacecolor=col,markersize=11,label=sl) for _,sl,col in scopes]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5,0.005), ncol=2, frameon=False, fontsize=11.5)
fig.subplots_adjust(top=0.90, bottom=0.22, wspace=0.1)
fig.savefig(R/"rq2_audit_f1_n.png", dpi=400, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
import pandas as pd
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES = PROJECT_ROOT / "outputs" / "rq2"
df = pd.read_csv(RES/"rq2_final_audit_240_MERGED.csv", encoding="utf-8-sig")
for c in ["pred_1A","pred_1B","pred_2","manual_1A","manual_1B","manual_2"]:
    df[c] = pd.to_numeric(df[c], errors="raise").astype(int)
df["pred_1B"] = ((df.pred_1A==1)&(df.pred_1B==1)).astype(int)

COMP = ["male_oriented_wellbeing","mainstream_self_improvement","unrelated_control"]
groups = [("Manosphere-adjacent", df.community_group=="manosphere_adjacent"),
          ("Comparison (pooled)", df.community_group.isin(COMP))]
rows = []
for gname, mask in groups:
    s = df[mask]
    for con in ["1A","1B","2"]:
        m,p = s[f"manual_{con}"], s[f"pred_{con}"]
        tp=int(((m==1)&(p==1)).sum()); fn=int(((m==1)&(p==0)).sum())
        fp=int(((m==0)&(p==1)).sum()); tn=int(((m==0)&(p==0)).sum())
        rows.append({"Community group":gname,"Construct":con,"Actual positives":tp+fn,
                     "TP":tp,"FN":fn,"FP":fp,"TN":tn,"N":len(s)})
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))
tbl.to_csv(RES/"audit_confusion_table.csv", index=False)